<a href="https://colab.research.google.com/github/Lingadarini/trendpulse/blob/part1-cleaning/part1_python_foundations/part1_python_foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone -b part1-cleaning https://github.com/Lingadarini/trendpulse.git
!ls trendpulse/part1_python_foundations/data

fatal: destination path 'trendpulse' already exists and is not an empty directory.
categories.csv	GBvideos.csv  INvideos.csv  USvideos.csv


# Part 1 - Python Foundations

### Notebook discipline

- The notebook has clear headings, one per task.
- It runs top to bottom with Runtime > Restart and run all, with no hidden state: every variable is created in a visible cell above where it is used.
- Configuration (paths, countries, thresholds) is declared once at the top.
- Scratch cells used for exploring the data were removed.

In [2]:
from datetime import datetime

# ---- Configuration ----
DATA_DIR = "trendpulse/part1_python_foundations/data/"   # str
COUNTRIES = ["US", "GB", "IN"]                           # list of str
MAX_ROWS = None                                          # None = load all rows
VIRAL_VIEWS = 1_000_000                                  # int (we may adjust later)
MIN_LIKE_RATIO = 0.90                                    # float (we may adjust later)
STRICT_MODE = True                                       # bool

# ---- Run header ----
start_time = datetime.now()
print("=" * 50)
print("TRENDPULSE - Part 1: Python Foundations")
print(f"Countries to load : {', '.join(COUNTRIES)}")
print(f"Max rows per file : {MAX_ROWS if MAX_ROWS else 'all'}")
print(f"Run started       : {start_time:%Y-%m-%d %H:%M:%S}")
print("=" * 50)

TRENDPULSE - Part 1: Python Foundations
Countries to load : US, GB, IN
Max rows per file : all
Run started       : 2026-11-04 01:43:53


In [3]:
import pandas as pd

REQUIRED_COLUMNS = ["video_id", "trending_date", "title", "channel_title",
                    "category_id", "publish_time", "views", "likes",
                    "dislikes", "comment_count", "comments_disabled",
                    "ratings_disabled"]

def load_country(path, country, max_rows=None):
    """Read one country CSV and return a list of dictionaries."""
    df = pd.read_csv(path, nrows=max_rows)

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{country}: missing columns {missing}")

    df["country"] = country
    return df.to_dict("records")


all_records = []
for country in COUNTRIES:
    path = f"{DATA_DIR}{country}videos.csv"
    rows = load_country(path, country, MAX_ROWS)
    print(f"{country}: {len(rows):,} records loaded")
    all_records.extend(rows)

print(f"TOTAL: {len(all_records):,} records")

# category_id -> category_name dictionary
cat_df = pd.read_csv(f"{DATA_DIR}categories.csv")
category_names = dict(zip(cat_df["category_id"], cat_df["category_name"]))
print(f"Categories loaded: {len(category_names)}")
print(category_names)

US: 16,080 records loaded
GB: 15,075 records loaded
IN: 15,075 records loaded
TOTAL: 46,230 records
Categories loaded: 17
{1: 'Film & Animation', 2: 'Autos & Vehicles', 10: 'Music', 15: 'Pets & Animals', 17: 'Sports', 19: 'Travel & Events', 20: 'Gaming', 22: 'People & Blogs', 23: 'Comedy', 24: 'Entertainment', 25: 'News & Politics', 26: 'Howto & Style', 27: 'Education', 28: 'Science & Technology', 29: 'Nonprofits & Activism', 30: 'Movies', 43: 'Shows'}


In [4]:
fail_counts = {"views": 0, "likes": 0, "dislikes": 0, "comment_count": 0,
               "comments_disabled": 0, "ratings_disabled": 0,
               "trending_date": 0, "publish_time": 0}

def to_int(value, field):
    """Convert to int. Return None and count a failure if impossible."""
    try:
        if pd.isna(value):
            raise ValueError("blank")
        return int(float(value))
    except (ValueError, TypeError):
        fail_counts[field] += 1
        return None

def to_bool(value, field):
    """Convert True/False-like values to a real boolean."""
    try:
        if isinstance(value, bool):
            return value
        text = str(value).strip().lower()
        if text in ("true", "1", "yes"):
            return True
        if text in ("false", "0", "no"):
            return False
        raise ValueError(text)
    except (ValueError, TypeError):
        fail_counts[field] += 1
        return False

def parse_trending_date(value):
    """'17.14.11' is yy.dd.mm -> 2017-11-14"""
    try:
        return datetime.strptime(str(value).strip(), "%y.%d.%m").date()
    except (ValueError, TypeError):
        fail_counts["trending_date"] += 1
        return None

def parse_publish_time(value):
    try:
        return datetime.fromisoformat(str(value).strip().replace("Z", ""))
    except (ValueError, TypeError):
        fail_counts["publish_time"] += 1
        return None


clean_records = []
dropped = 0

for rec in all_records:
    views = to_int(rec["views"], "views")
    if views is None:          # rule: bad views -> drop the record
        dropped += 1
        continue

    likes = to_int(rec["likes"], "likes")
    dislikes = to_int(rec["dislikes"], "dislikes")
    comments = to_int(rec["comment_count"], "comment_count")

    clean_records.append({
        "video_id": rec["video_id"],
        "country": rec["country"],
        "title": rec["title"],
        "channel_title": rec["channel_title"],
        "category_id": rec["category_id"],
        "views": views,
        "likes": likes if likes is not None else 0,
        "dislikes": dislikes if dislikes is not None else 0,
        "comment_count": comments if comments is not None else 0,
        "comments_disabled": to_bool(rec["comments_disabled"], "comments_disabled"),
        "ratings_disabled": to_bool(rec["ratings_disabled"], "ratings_disabled"),
        "trending_date": parse_trending_date(rec["trending_date"]),
        "publish_time": parse_publish_time(rec["publish_time"]),
    })

print(f"Records kept   : {len(clean_records):,}")
print(f"Records dropped: {dropped}")
print("Parse failures per field:")
for field, n in fail_counts.items():
    print(f"  {field:20s} {n}")
print()
print(clean_records[0])

Records kept   : 46,230
Records dropped: 0
Parse failures per field:
  views                0
  likes                12
  dislikes             0
  comment_count        36
  comments_disabled    0
  ratings_disabled     0
  trending_date        0
  publish_time         0

{'video_id': 'g_N-w8LeAi8', 'country': 'US', 'title': 'Honest Football Prank LEAKED', 'channel_title': 'QuietSpeech10', 'category_id': 22, 'views': 48555, 'likes': 2189, 'dislikes': 180, 'comment_count': 136, 'comments_disabled': False, 'ratings_disabled': False, 'trending_date': datetime.date(2017, 11, 14), 'publish_time': datetime.datetime(2017, 11, 14, 7, 4, 28)}


In [5]:
# 1. Tuples: fixed key fields that should not change after cleaning
key_tuples = []
for rec in clean_records:
    key_tuples.append((rec["video_id"], rec["trending_date"], rec["views"], rec["likes"]))
print("First tuple:", key_tuples[0])

# 2. Sets: unique video IDs per country
unique_ids = {}
for country in COUNTRIES:
    ids = set()
    for rec in clean_records:
        if rec["country"] == country:
            ids.add(rec["video_id"])
    unique_ids[country] = ids
    print(f"{country}: {len(ids):,} unique videos")

in_all_three = unique_ids["US"] & unique_ids["GB"] & unique_ids["IN"]
us_only = unique_ids["US"] - (unique_ids["GB"] | unique_ids["IN"])
print(f"Trended in all 3 countries: {len(in_all_three):,}")
print(f"Trended only in US        : {len(us_only):,}")

# 3. Dictionary counter: trending appearances per channel
channel_counts = {}
for rec in clean_records:
    name = rec["channel_title"]
    channel_counts[name] = channel_counts.get(name, 0) + 1

# 4. Slicing: top 15 channels
ranked = sorted(channel_counts.items(), key=lambda item: item[1], reverse=True)
print("\nTop 15 channels:")
for name, count in ranked[:15]:
    print(f"  {name:30s} {count}")

# 5. Nested dictionary: country -> category name -> record count
summary = {}
for rec in clean_records:
    cat_name = category_names.get(rec["category_id"], "Unknown")
    summary.setdefault(rec["country"], {})
    summary[rec["country"]][cat_name] = summary[rec["country"]].get(cat_name, 0) + 1

print("\nSample of nested summary:")
for country in COUNTRIES:
    for cat_name, n in list(summary[country].items())[:3]:
        print(f"  {country} | {cat_name:20s} | {n:,}")

First tuple: ('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)
US: 5,068 unique videos
GB: 4,113 unique videos
IN: 5,878 unique videos
Trended in all 3 countries: 20
Trended only in US        : 4,834

Top 15 channels:
  QuietChallenge75               2464
  OfficialReview41               1586
  UltimateUnboxing65             1433
  InsaneReaction46               1233
  Brand NewChallenge45           1175
  CrazyTrailer91                 1018
  FinalTrailer17                 911
  UltimateSpeech20               770
  Brand NewSecret51              684
  WildHighlights7                682
  SurprisingHighlights41         650
  FinalBreakdown11               643
  CrazyRecipe67                  603
  QuietInterview9                570
  UltimateVlog96                 565

Sample of nested summary:
  US | People & Blogs       | 1,423
  US | News & Politics      | 755
  US | Music                | 3,052
  GB | Science & Technology | 470
  GB | Howto & Style        | 588
  GB | Comed

### Why each data structure was the right choice

| Structure | Used for | Why it fits | Mutable? |
|---|---|---|---|
| List of dictionaries | The cleaned records | Keeps the order of rows, and each record's fields can be read by name (rec["views"]) | Mutable |
| Tuple | Key fields (video_id, trending_date, views, likes) | These values describe a finished observation and should not change by accident. Tuples are fixed after creation | Immutable |
| Set | Unique video IDs per country | Removes duplicates automatically and supports fast intersection (&) and difference (-) to find cross-country videos | Mutable |
| Dictionary counter | Trending appearances per channel | Looks up a channel name directly and updates its count in one step | Mutable |
| Nested dictionary | country -> category -> record count | Matches the natural hierarchy of the question: country first, then category | Mutable |
| Slicing | Top 15 channels (ranked[:15]) | Takes the first 15 items of the sorted list without writing a loop | Works on lists and tuples |

## Variable scope

In [6]:
outside_var = "I live outside the function"

def scope_demo():
    inside_var = "I live inside the function"
    print("Inside function, can see outside_var:", outside_var)
    print("Inside function, can see inside_var :", inside_var)

scope_demo()

try:
    print(inside_var)
except NameError as e:
    print("Outside function, cannot see inside_var ->", e)

Inside function, can see outside_var: I live outside the function
Inside function, can see inside_var : I live inside the function
Outside function, cannot see inside_var -> name 'inside_var' is not defined


**Why this happens:** `inside_var` is created inside the function, so it is local and exists only while the function runs. Reading it outside gives a `NameError`. `outside_var` is created outside any function, so it is global and the function can read it.

## Loops over strings and sequences

In [7]:
def title_features(title):
    """Count uppercase words, ! and ? marks, and words in a title."""
    uppercase_words = 0
    punctuation = 0

    for ch in title:                  # loop over characters
        if ch == "!" or ch == "?":
            punctuation += 1

    words = title.split()
    for word in words:                # loop over words
        if word.isupper() and len(word) > 1:
            uppercase_words += 1

    return {"uppercase_words": uppercase_words,
            "punctuation_marks": punctuation,
            "word_count": len(words)}

print(title_features("Honest Football Prank LEAKED!"))

# range() with index access
for i in range(3):
    print(i, clean_records[i]["title"])

# while loop with break and continue: first video above the threshold
i = 0
found = None
while i < len(clean_records):
    rec = clean_records[i]
    i += 1
    if rec["views"] is None:
        continue                      # skip bad records
    if rec["views"] > VIRAL_VIEWS:
        found = rec
        break                         # stop at the first match
print("First video above", f"{VIRAL_VIEWS:,}", "views:", found["title"] if found else "none")

{'uppercase_words': 1, 'punctuation_marks': 1, 'word_count': 4}
0 Honest Football Prank LEAKED
1 Surprising Cricket Review
2 Mira Solis - Football (Live) OFFICIAL
First video above 1,000,000 views: Official Cats Speech EXCLUSIVE


In [8]:
all_views = sorted(rec["views"] for rec in clean_records)
n = len(all_views)

print("Smallest views :", f"{all_views[0]:,}")
print("25th percentile:", f"{all_views[n // 4]:,}")
print("Median         :", f"{all_views[n // 2]:,}")
print("75th percentile:", f"{all_views[3 * n // 4]:,}")
print("90th percentile:", f"{all_views[9 * n // 10]:,}")
print("Largest views  :", f"{all_views[-1]:,}")

zero_views = sum(1 for rec in clean_records if rec["views"] == 0)
no_ratings = sum(1 for rec in clean_records if rec["ratings_disabled"])
print("Records with 0 views      :", zero_views)
print("Records with ratings off  :", no_ratings)

Smallest views : 1,066
25th percentile: 129,462
Median         : 318,278
75th percentile: 824,051
90th percentile: 2,025,449
Largest views  : 360,760,164
Records with 0 views      : 0
Records with ratings off  : 544


## Rule-based classification

In [9]:
HIT_VIEWS = 2_000_000    # Viral Hit: about the top 10% of views
LOW_VIEWS = 130_000      # below this: Low Engagement (about the bottom 25%)

def classify_video(record):
    """Return one label for a record using views and like ratio."""
    views = record["views"]
    likes = record["likes"]
    dislikes = record["dislikes"]

    # Edge cases first, so we never divide by zero
    if record["ratings_disabled"]:
        return "Unrated"
    total_votes = likes + dislikes
    if total_votes == 0:
        return "Unrated"

    like_ratio = likes / total_votes

    if views >= HIT_VIEWS:
        if like_ratio >= MIN_LIKE_RATIO:
            return "Viral Hit"
        else:
            return "Divisive"
    elif views >= LOW_VIEWS:
        if like_ratio >= MIN_LIKE_RATIO:
            return "Steady Performer"
        else:
            return "Divisive"
    else:
        return "Low Engagement"


label_counts = {}
for rec in clean_records:
    label = classify_video(rec)
    label_counts[label] = label_counts.get(label, 0) + 1

print("Records per label:")
for label, n in label_counts.items():
    print(f"  {label:18s} {n:,}")
print("Total:", f"{sum(label_counts.values()):,}")

Records per label:
  Low Engagement     11,460
  Divisive           4,202
  Steady Performer   25,669
  Viral Hit          4,355
  Unrated            544
Total: 46,230


### Thresholds used

| Setting | Value | Reason |
|---|---|---|
| HIT_VIEWS | 2,000,000 | About the 90th percentile of views in this data |
| LOW_VIEWS | 130,000 | About the 25th percentile of views |
| MIN_LIKE_RATIO | 0.90 | Videos below this like share are treated as Divisive |

**Edge cases handled:** if ratings are disabled, or likes + dislikes is 0, the record is labelled `Unrated` before any division happens, so there is no division-by-zero error.

### Hand-trace of one record

Take a record with views = 2,500,000, likes = 90,000, dislikes = 5,000, ratings_disabled = False.

1. ratings_disabled is False, so the first `if` is skipped.
2. total_votes = 90,000 + 5,000 = 95,000, which is not 0, so the second `if` is skipped.
3. like_ratio = 90,000 / 95,000 = 0.947.
4. views (2,500,000) >= HIT_VIEWS (2,000,000), so we enter the first branch.
5. Inside it, like_ratio (0.947) >= 0.90, so the result is **Viral Hit**.

## Refactor

In [10]:
def count_by(records, key_function):
    """Count how many records fall under each key."""
    counts = {}
    for rec in records:
        key = key_function(rec)
        counts[key] = counts.get(key, 0) + 1
    return counts

# The same counting block, now reused:
channel_counts_v2 = count_by(clean_records, lambda r: r["channel_title"])
label_counts_v2 = count_by(clean_records, classify_video)

print("Labels match:", label_counts_v2 == label_counts)
print("Channels match:", channel_counts_v2 == channel_counts)

Labels match: True
Channels match: True


**What we gained:** the counting block was written three times (channels, labels, countries/categories). Now it lives in one function, `count_by`, so a bug fix or change happens in one place. It also makes new counts a one-line call.